In [1]:
import sys
sys.path.append(r"C:\Users\User\Downloads\cds\CDS")

import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from tqdm import tqdm
from src.utils import evaluate, get_device


In [2]:
# =====================
# Load data
# =====================
train_metadata_csv = r"C:\Users\User\Downloads\cds\CDS\processed_train_metadata.csv"
test_metadata_csv  = r"C:\Users\User\Downloads\cds\CDS\processed_test_metadata.csv"

TRAIN_AUDIO_DIR = r"C:\Users\User\Downloads\cds\CDS\processed_train_audio"
TEST_AUDIO_DIR  = r"C:\Users\User\Downloads\cds\CDS\processed_test_audio"

train_df = pd.read_csv(train_metadata_csv)[["video_id", "emotion"]]
test_df  = pd.read_csv(test_metadata_csv)[["video_id", "emotion"]]

# =====================
# Emotion mapping
# =====================
EMOTION_MAP = {
    'neutral': 'Neutral',   'calm': 'Neutral',
    'surprise': 'Surprised', 'surprised': 'Surprised',
    'fear': 'Fearful',      'fearful': 'Fearful',
    'sad': 'Sad',           'sadness': 'Sad',
    'happy': 'Happy',       'joy': 'Happy',
    'disgust': 'Disgust',
    'anger': 'Anger',       'angry': 'Anger'
}

train_df["emotion"] = train_df["emotion"].map(EMOTION_MAP)
test_df["emotion"]  = test_df["emotion"].map(EMOTION_MAP)

# =====================
# Drop rows with missing audio files
# =====================
train_df = train_df[
    train_df["video_id"].apply(
        lambda vid: os.path.exists(os.path.join(TRAIN_AUDIO_DIR, vid + ".wav"))
    )
].reset_index(drop=True)

test_df = test_df[
    test_df["video_id"].apply(
        lambda vid: os.path.exists(os.path.join(TEST_AUDIO_DIR, vid + ".wav"))
    )
].reset_index(drop=True)

# =====================
# Label encoding
# =====================
label_list = sorted(train_df["emotion"].unique())
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}

In [3]:
# =====================
# Dataset
# =====================
class MELDAudioDataset(Dataset):
    def __init__(self, df, audio_dir, label2id, sample_rate=16000, n_mels=64, max_length=128):
        self.df          = df.reset_index(drop=True)
        self.audio_dir   = audio_dir
        self.label2id    = label2id
        self.sample_rate = sample_rate
        self.n_mels      = n_mels
        self.max_length  = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row["video_id"] + ".wav")
        y, sr      = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        mel_spec   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=self.n_mels)
        mel_spec   = librosa.power_to_db(mel_spec, ref=np.max)
        mel_tensor = torch.from_numpy(mel_spec.astype(np.float32)).unsqueeze(0)
        if mel_tensor.shape[-1] < self.max_length:
            mel_tensor = torch.nn.functional.pad(mel_tensor, (0, self.max_length - mel_tensor.shape[-1]))
        else:
            mel_tensor = mel_tensor[..., :self.max_length]
        label = self.label2id[row["emotion"]]
        return mel_tensor, label


In [4]:
# =====================
# Model
# =====================
class AudioCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.pool       = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

In [5]:
# =====================
# DataLoaders
# =====================
train_dataset = MELDAudioDataset(train_df, TRAIN_AUDIO_DIR, label2id)
test_dataset  = MELDAudioDataset(test_df,  TEST_AUDIO_DIR,  label2id)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32)

In [6]:
# =====================
# Training
# =====================
device    = get_device()
model     = AudioCNN(num_classes=len(label_list)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs    = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for mel_spec, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        mel_spec, labels = mel_spec.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs    = model(mel_spec)
        loss       = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1} - Avg Loss: {train_loss / len(train_loader):.4f}")

Epoch 1/10:   0%|          | 0/313 [00:00<?, ?it/s]c:\Users\User\Downloads\cds\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Epoch 1/10:  24%|██▍       | 76/313 [00:50<02:09,  1.83it/s]c:\Users\User\Downloads\cds\venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1365
  warnings.warn(
Epoch 1/10:  31%|███       | 97/313 [01:00<01:46,  2.03it/s]c:\Users\User\Downloads\cds\venv\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1024
  warnings.warn(
Epoch 1/10: 100%|██████████| 313/313 [03:13<00:00,  1.62it/s]


Epoch 1 - Avg Loss: 1.5437


Epoch 2/10: 100%|██████████| 313/313 [02:08<00:00,  2.43it/s]


Epoch 2 - Avg Loss: 1.5208


Epoch 3/10: 100%|██████████| 313/313 [02:34<00:00,  2.03it/s]


...

Epoch 8 - Avg Loss: 1.4855


Epoch 9/10: 100%|██████████| 313/313 [01:17<00:00,  4.02it/s]


Epoch 9 - Avg Loss: 1.4802


Epoch 10/10: 100%|██████████| 313/313 [01:02<00:00,  4.98it/s]

Epoch 10 - Avg Loss: 1.4797


In [7]:
# =====================
# Evaluation
# =====================
model.eval()
preds, true_labels = [], []
with torch.no_grad():
    for mel_spec, labels in test_loader:
        mel_spec = mel_spec.to(device)
        outputs  = model(mel_spec)
        preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        true_labels.extend(labels.numpy())

evaluate(true_labels, preds)
print(classification_report(true_labels, preds, target_names=label_list))

Validation Accuracy: 0.4801
Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.24      0.27       345
           1       0.00      0.00      0.00        68
           2       0.00      0.00      0.00        50
           3       0.16      0.06      0.09       402
           4       0.53      0.90      0.66      1256
           5       0.00      0.00      0.00       208
           6       0.41      0.06      0.11       281

    accuracy                           0.48      2610
   macro avg       0.20      0.18      0.16      2610
weighted avg       0.36      0.48      0.38      2610

Confusion Matrix:
[[  82    0    0   30  225    0    8]
 [  14    0    0    3   51    0    0]
 [   5    0    0    8   36    0    1]
 [  47    0    0   25  324    0    6]
 [  71    0    0   47 1128    0   10]
 [  18    0    0   12  177    0    1]
 [  28    0    0   28  207    0   18]]
              precision    recall  f1-score   support

       Anger

c:\Users\User\Downloads\cds\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\Downloads\cds\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\Downloads\cds\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\U

In [8]:
torch.save(model.state_dict(), r"C:\Users\User\Downloads\cds\CDS\audio_model_weights.pth")
print("saved")

saved
